# The worst errors

What do the homes with the largest errors have in common? The answer tells us what to do next:
- **Bad data:** add a cleaning rule.
- **Something the model cannot see** (for example, the condition of the home): we can add a feature only if the information is known before listing.
- **Areas with few nearby sales:** give a wider price range for these homes.
- **Nothing in common:** the errors are noise.

**Data.** The validation predictions of the selected model (XGBoost, 06 settings, school districts + HOA + flooring + comparable sales, log price), saved by `07_final_model_selection.ipynb`. Each validation month (2025-10 to 2026-03) is predicted by a model trained on the 18 months before it, so each prediction is out of sample. This notebook does not use the test month (2026-04).

**Note:** `ListPrice` and `DaysOnMarket` are known only after listing. We use them here to understand the errors, not as model inputs.

In [1]:
import numpy as np
import pandas as pd

from idx_exchange_data_science.cleaning import TARGET
from idx_exchange_data_science.features import build_table
from idx_exchange_data_science.paths import DATA_DIR, FILTERED_PATH

pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.width", 200)

MODEL = "XGBoost (06 settings) | Districts + HOA + flooring + comps | log price"
predictions = pd.read_parquet(DATA_DIR / "experiments" / "final_model_selection_predictions.parquet")[MODEL]

df = build_table()
extra = pd.read_csv(FILTERED_PATH, usecols=["ListingKey", "UnparsedAddress", "ListPrice", "DaysOnMarket"], low_memory=False)
df = df.merge(extra.drop_duplicates("ListingKey"), on="ListingKey", how="left", validate="many_to_one")

v = df.loc[predictions.index].assign(estimate=predictions)
v["ratio"] = v["estimate"] / v[TARGET]
v["error"] = (v["ratio"] - 1).abs()
v["sale_to_list"] = v[TARGET] / v["ListPrice"]
v["ppsf_vs_comps"] = (v[TARGET] / v["LivingArea"]) / np.exp(v["LogCompPricePerSqft"])
cutoff = v["error"].quantile(0.95)
worst = v["error"] >= cutoff
print(f"{len(v):,} validation homes. The worst 5% have an error of {cutoff:.1%} or more and give "
      f"{v.loc[worst, 'error'].sum() / v['error'].sum():.0%} of the total error.")

59,363 validation homes. The worst 5% have an error of 31.6% or more and give 23% of the total error.


## 1. Direction and price level

In [2]:
pd.DataFrame({
    "worst 5%": [(v.loc[worst, "ratio"] > 1).mean(), v.loc[worst, TARGET].median(), v.loc[worst, "estimate"].median(),
                 v.loc[worst, "sale_to_list"].median(), v.loc[worst, "ppsf_vs_comps"].median()],
    "other homes": [(v.loc[~worst, "ratio"] > 1).mean(), v.loc[~worst, TARGET].median(), v.loc[~worst, "estimate"].median(),
                    v.loc[~worst, "sale_to_list"].median(), v.loc[~worst, "ppsf_vs_comps"].median()],
}, index=["share over-estimated", "median sale price", "median estimate", "median sale / list price",
          "median $ per sq ft / comparables"])

,worst 5%,other homes
share over-estimated,0.818,0.498
median sale price,"870,000.000","875,000.000"
median estimate,"1,097,765.000","881,896.250"
median sale / list price,0.965,1.000
median $ per sq ft / comparables,0.719,0.994


**Observations**
- **The worst 5% have an error of 31.6% or more.** They give 23% of the total error.
- **82% of them are over-estimates** (other homes: 50%). The model is too high for these homes.
- **They sold near their own list price** (median 96.5% of list). Thus, the seller also priced them low. The market did not surprise the seller; the model does not see something about the home.
- **They sold at 0.72x the price per sq ft of their comparable sales** (other homes: 0.99x). Typical causes are a home in poor condition, a teardown, a forced sale or a land lease. The data has no field for these.

## 2. What is more common in the worst 5%
"Lift" is the share in the worst 5% divided by the share in the other homes. A lift of 3 means three times more common.

In [3]:
california_zip = pd.to_numeric(v["PostalCode"], errors="coerce").between(90001, 96162)
address = v["UnparsedAddress"].fillna("").str.lower()
signals = {
    "ZIP code outside California": ~california_zip,
    "mobile-home space (Spc or Space in the address)": address.str.contains(r"\bspc\b|\bspace\b", regex=True),
    "no comparable sales": v["LogCompPricePerSqft"].isna(),
    "comparable sales more than 5 km away": v["CompDistanceKm"] > 5,
    "lot larger than 1 acre": v["LotSizeSquareFeet"] > 43_560,
    "living area under 600 sq ft": v["LivingArea"] < 600,
    "listed more than 6 months before the sale": v["MonthsToClose"] > 6,
    "new construction": v["NewConstructionYN"] == 1,
    "sold below 0.5x the comparables' $ per sq ft": v["ppsf_vs_comps"] < 0.5,
    "sold above 2x the comparables' $ per sq ft": v["ppsf_vs_comps"] > 2,
    "sold below 70% of list price (after listing)": v["sale_to_list"] < 0.7,
}
lift = pd.DataFrame({name: {"homes": int(s.sum()), "share in worst 5%": s[worst].mean(), "share in others": s[~worst].mean()}
                     for name, s in signals.items()}).T
lift["lift"] = lift["share in worst 5%"] / lift["share in others"]
lift.sort_values("lift", ascending=False)

,homes,share in worst 5%,share in others,lift
sold below 0.5x the comparables' $ per sq ft,283.000,0.080,0.001,97.862
sold below 70% of list price (after listing),59.000,0.016,0.000,74.394
ZIP code outside California,9.000,0.002,0.000,23.743
mobile-home space (Spc or Space in the address),2.000,0.000,0.000,18.994
sold above 2x the comparables' $ per sq ft,250.000,0.039,0.002,16.443
comparable sales more than 5 km away,464.000,0.037,0.006,5.902
no comparable sales,161.000,0.011,0.002,4.897
lot larger than 1 acre,"3,438.000",0.168,0.052,3.217
living area under 600 sq ft,159.000,0.008,0.002,3.212
listed more than 6 months before the sale,"3,717.000",0.121,0.060,2.024


In [4]:
show = ["UnparsedAddress", "City", "CountyOrParish", TARGET, "estimate", "ListPrice", "LivingArea", "LotSizeSquareFeet",
        "CompDistanceKm", "ppsf_vs_comps"]
v.loc[worst].sort_values("error", ascending=False)[show].head(15)

,UnparsedAddress,City,CountyOrParish,ClosePrice,estimate,ListPrice,LivingArea,LotSizeSquareFeet,CompDistanceKm,ppsf_vs_comps
268392,39450 Tassajara Road,Carmel Valley,Monterey,"479,500.000","2,064,743.125","550,000.000","3,000.000","1,745,449.000",NaN,NaN
248176,5600 Merriewood Dr,Oakland,Alameda,"200,000.000","824,989.375","225,000.000","1,023.000","6,785.000",0.254,0.248
236991,5680 San Felipe,San Jose,Santa Clara,"975,000.000","3,503,572.500","999,900.000","3,894.000","61,420.000",0.685,0.254
270469,6011 Main,South Gate,Los Angeles,"245,000.000","804,695.812","245,000.000","1,605.000","5,519.000",0.290,0.272
268335,"239 73 Arena Brillante, Ensenada, Baja, Mexico",Other,Foreign Country,"200,000.000","653,869.812","162,500.000","1,400.000","1,400.000",NaN,NaN
292573,3946 Tunitas Creek Road,Half Moon Bay,San Mateo,"915,000.000","2,932,714.750","750,000.000","1,400.000","2,265,120.000",NaN,NaN
268352,125 Ondinas,Other,Other State,"200,000.000","621,415.562","240,000.000","1,862.000","1,345.000",NaN,NaN
290356,5917 Road 30,Unincorporated,Tulare,"190,000.000","543,196.812","200,000.000",836.000,"871,200.000",23.649,1.061
274190,8016 N Lake Blvd,Kings Beach,Placer,"580,000.000","1,651,439.875","595,000.000","2,167.000","11,119.000",15.913,0.414
269820,427 S Gramercy Place,Los Angeles,Los Angeles,"1,200,000.000","3,409,719.750","1,295,000.000","5,814.000","8,010.000",0.469,0.263


**Observations**
- **Price far below the comparables is the strongest signal.** Homes that sold below 0.5x the comparables' price per sq ft are 98 times more common in the worst 5%. But this is known only after the sale, so it cannot be a feature.
- **Few or distant comparable sales:** comparables more than 5 km away are 5.9 times more common, and no comparables 4.9 times. These homes are in rural areas with few sales.
- **Large lots:** lots above 1 acre are 3.2 times more common (17% of the worst 5%).
- **A small number of data problems:** listings outside California (for example, Ensenada, Mexico) and mobile homes on a rented space ("Spc 207"). Some homes in the list are probably on leased land (for example, a Newport Beach home with a 21-acre lot).
- **New construction is less common** in the worst 5% (lift 0.3).

## 3. Data problems in the full dataset
How many sales in all months have the data problems that section 2 found?

In [5]:
zip_number = pd.to_numeric(df["PostalCode"], errors="coerce")
full_address = df["UnparsedAddress"].fillna("").str.lower()
problems = {
    "ZIP code outside California (or missing)": ~zip_number.between(90001, 96162),
    "county is Foreign Country, Other State or Other": df["CountyOrParish"].isin(["Foreign Country", "Other State", "Other"]),
    "mobile-home space (Spc or Space in the address)": full_address.str.contains(r"\bspc\b|\bspace\b", regex=True),
}
pd.DataFrame({name: {"sales": int(m.sum()), "share": m.mean()} for name, m in problems.items()}).T

,sales,share
ZIP code outside California (or missing),27.000,0.000
"county is Foreign Country, Other State or Other",6.000,0.000
mobile-home space (Spc or Space in the address),6.000,0.000


In [6]:
outside = ~zip_number.between(90001, 96162)
df.loc[outside, ["UnparsedAddress", "City", "PostalCode", "CountyOrParish", "Latitude", TARGET]].head(10)

,UnparsedAddress,City,PostalCode,CountyOrParish,Latitude,ClosePrice
21978,5461 Trinette Avenue,Garden Grove,98245,Orange,33.782,"1,150,000.000"
65891,25820 Jumano Dr.,Moreno Valley,82553,Riverside,33.895,"570,000.000"
74035,4134 Stone Valley Ct,Chico,05073,Butte,39.791,"580,000.000"
89689,1103 Maynard,Duarte,01010,Los Angeles,36.778,"789,000.000"
141509,19649 DRYCLIFF Street,Canyon Country,19351,Los Angeles,34.422,"825,000.000"
164728,"191 Espana, Ensenada, Baja, Mexico",Outside Area (Outside Ca),22870,Other State,NaN,"421,000.000"
165609,15948 Conservatory Drive,Fontana,02336,San Bernardino,34.172,"1,137,519.000"
183062,7510 Tuckaway Harbor St,Outside Area (Outside Ca),89139,Nevada,36.053,"442,000.000"
194857,2673 Hannah Creek Lane,Rialto,20205,San Bernardino,34.149,"681,020.000"
206831,7387 Carrollton Pl,Corona,88888,Other,33.954,"960,000.000"


**Observations**
- **Only 27 sales (0.01%) have a ZIP code outside California, and most of them are California homes with an incorrect ZIP code** (for example, Garden Grove with 98245). A rule that removes these sales would remove valid homes. A better rule: set an invalid ZIP code to missing. The coordinate fill and the target encoding then use the county.
- **Approximately 6 sales are outside California** (county "Foreign Country", "Other State" or "Other", or city "Outside Area (Outside Ca)"). A rule can remove them.
- **6 sales are mobile-home spaces.**
- These problems are too rare to change the validation scores. They are worth fixing for correctness, but Dataset A is the team's agreed dataset, so the team must agree to the change.

## 4. Error by distance to the comparable sales and by lot size
The price range in the app is the same width (relative to the estimate) for every home. If the error depends on something that is known before listing, the range should depend on it too.

In [7]:
def error_table(groups):
    return v.groupby(groups, observed=True)["error"].agg(
        homes="size", median="median", p80=lambda e: e.quantile(0.8), p90=lambda e: e.quantile(0.9))


distance = pd.cut(v["CompDistanceKm"], [0, 0.5, 1, 2, 5, np.inf], labels=["under 0.5 km", "0.5 to 1 km", "1 to 2 km", "2 to 5 km", "over 5 km"])
error_table(distance)

,homes,median,p80,p90
CompDistanceKm,,,,
under 0.5 km,47441,0.069,0.151,0.220
0.5 to 1 km,8224,0.080,0.172,0.251
1 to 2 km,2088,0.102,0.228,0.332
2 to 5 km,915,0.129,0.283,0.413
over 5 km,464,0.142,0.339,0.497


In [8]:
lot = pd.cut(v["LotSizeSquareFeet"], [0, 5_000, 10_000, 43_560, 435_600, np.inf],
             labels=["under 5,000 sq ft", "5,000 to 10,000", "10,000 sq ft to 1 acre", "1 to 10 acres", "over 10 acres"])
error_table(lot)

,homes,median,p80,p90
LotSizeSquareFeet,,,,
"under 5,000 sq ft",9500,0.068,0.155,0.229
"5,000 to 10,000",33105,0.066,0.142,0.206
"10,000 sq ft to 1 acre",12132,0.085,0.186,0.266
1 to 10 acres,3037,0.115,0.250,0.362
over 10 acres,401,0.173,0.381,0.517


**Observations**
- **The error depends on the distance to the comparable sales.** The median error increases from 6.9% (comparables within 0.5 km, 80% of homes) to 14.2% (more than 5 km). The 90th percentile increases from 22% to 50%.
- **The error also depends on the lot size:** 6.6% to 6.8% for lots below 10,000 sq ft, 11.5% for 1 to 10 acres and 17.3% above 10 acres.
- **Thus, one range width for all homes is not correct.** The 80% range in the app is too wide for a typical home near many sales, and too narrow for a rural home. Both values are known before listing, so the app can give a range for each group (group-wise conformal prediction).

## Summary
- **Most large errors come from something the model cannot see:** the home sold near its list price, but far below the price per sq ft of nearby sales (poor condition, teardown, forced sale, land lease). No field before listing shows this. The list price would show it, but the model must predict before listing.
- **Rural homes with few nearby sales and large lots have larger errors.** This is a confidence problem, not a data problem.
- **Recommended changes:**
  1. **Price ranges by group:** calculate the ranges separately for each comparable-distance group (and possibly lot size). Check the coverage on held-out months.
  2. **Cleaning rules (for the team to agree):** remove the approximately 6 sales outside California and the 6 mobile-home spaces, and set invalid ZIP codes to missing. The effect on the scores is very small.
  3. **No new feature** is available for the condition of the home before listing.